# 3. Model Gabungan -- Bantul dan Sleman dalam Satu Model

Notebook ketiga dari empat notebook **Gabungan**.

Dua notebook wilayah (`Bantul/Script penyakitan/Stacked Revisi.ipynb` dan
padanannya di Sleman) masing-masing melatih empat model untuk empat penyakit di
wilayahnya sendiri. Notebook ini melatih **satu set model di atas data kedua
wilayah sekaligus**, dengan satu tambahan penting:

> **`wilayah` menjadi fitur kategorik ketujuh.**

Itulah yang membuat notebook ini berbeda dari sekadar "melatih ulang dengan data
lebih banyak". Dengan `wilayah` di dalam model, kita bisa bertanya:

> Setelah usia, jenis kelamin, segmen kepesertaan, tinggi, berat dan IMT
> diperhitungkan, **apakah kabupaten tempat seseorang diskrining masih
> menjelaskan sesuatu tentang risikonya?**

Notebook `1. Penggabungan dan Perbandingan Populasi` sudah menunjukkan bahwa
prevalensi kedua wilayah berbeda dan komposisi segmennya berbeda jauh. Yang belum
terjawab: apakah selisih prevalensi itu **tersisa** setelah komposisi
diperhitungkan, atau habis dijelaskan olehnya. Kepentingan fitur `wilayah` di
Bagian 6 adalah jawabannya.

### Metodologi dijaga identik dengan notebook wilayah

Supaya angkanya bisa dibandingkan langsung dengan `ringkasan_model.csv` tiap
wilayah, semua pilihan metodologi dipertahankan sama persis:

| Aspek | Nilai |
|---|---|
| Model dibandingkan | `dummy` - `logistic` - `decision_tree` - `xgboost` |
| Pembagian data | 80:20, terstratifikasi |
| Validasi silang | `StratifiedKFold(5, shuffle=True)` |
| Pencarian hyperparameter | `RandomizedSearchCV`, `n_iter=30` |
| Kriteria pencarian | `scoring="roc_auc"` |
| Penanganan ketimpangan | `class_weight`, hanya bila kelas positif minoritas |
| `random_state` | 42, di ketujuh tempat yang berpengaruh |

**Random Forest tidak dipakai.** Ia pernah dievaluasi lalu dikeluarkan dari
proyek atas keputusan pengguna; menambahkannya kembali di sini akan membuat
notebook Gabungan tidak sebanding dengan notebook wilayah.

⚠️ **`dummy` tidak pernah boleh terpilih sebagai "model terbaik".** Pada
Hipertensi kelas positif adalah mayoritas, sehingga `dummy` mencetak F1 lebih
tinggi daripada model sungguhan mana pun. Semua pemilihan model di notebook ini
lewat satu fungsi `model_terbaik()` yang mengecualikannya.

In [ ]:
# ===================== IMPORTS =====================
import os
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     RandomizedSearchCV)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             ConfusionMatrixDisplay)
from xgboost import XGBClassifier

AKAR = Path.cwd()
while not (AKAR / "Bersama").exists():
    AKAR = AKAR.parent
sys.path.insert(0, str(AKAR / "Bersama"))

import konfigurasi as cfg
import viz_style as vz

vz.apply_style()
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 190)

RANDOM_STATE = 42
WARNA = vz.PALETTE["WILAYAH"]

In [ ]:
# ===================== KONFIGURASI =====================
DATA_DIR = cfg.folder_data("Gabungan")

# ---------------------------------------------------------------------
# QUICK_MODE = True  -> 8 kandidat, CV 3 lipat   (pengembangan)
# QUICK_MODE = False -> 30 kandidat, CV 5 lipat  (LAPORAN FINAL)
# ---------------------------------------------------------------------
QUICK_MODE = False

N_ITER = 8 if QUICK_MODE else 30
CV_FOLDS = 3 if QUICK_MODE else 5
CV = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

DISEASES = {
    "Diabetes Melitus": ("dataset_diabetes.csv", "Berisiko Diabetes Melitus", "15+"),
    "Hipertensi":       ("dataset_hipertensi.csv", "Berisiko Hipertensi", "15+"),
    "Kanker Paru":      ("dataset_kanker_paru.csv", "Berisiko Kanker Paru", "45+"),
    "TBC":              ("dataset_tbc.csv", "Berisiko TBC", "semua usia"),
}

print("=" * 72)
if QUICK_MODE:
    print("QUICK_MODE = True   ->  n_iter=8, cv=3")
    print()
    print("  PERINGATAN: hasil SEMENTARA, belum layak dilaporkan.")
    print("  Setel QUICK_MODE = False lalu jalankan ulang untuk angka final.")
else:
    print("QUICK_MODE = False  ->  n_iter=30, cv=5   (mode laporan final)")
print("=" * 72)
print(f"Sumber data: {DATA_DIR}")

## 1. Memuat Data Gabungan

Berkas yang dibaca dihasilkan notebook `1. Penggabungan dan Perbandingan
Populasi`. Setiap berkas sudah membawa kolom `wilayah`.

Perhatikan bahwa keempat dataset punya **jumlah baris dan pita usia yang
berbeda**, karena `Tidak Dapat Pertanyaan` adalah gerbang kelayakan kuesioner.
Setiap metrik di notebook ini karena itu selalu ditampilkan bersama pita usianya.

In [ ]:
# ===================== MUAT DATASET =====================
data = {}
for nama, (berkas, target, pita) in DISEASES.items():
    d = pd.read_csv(DATA_DIR / berkas)
    X = d.drop(columns=[target])
    y = d[target].astype(int)
    data[nama] = {"X": X, "y": y, "usia": pita, "target": target,
                  "base_rate": float(y.mean())}
    print(f"{nama:<18} {len(d):>9,} baris | pita {pita:<11} | "
          f"prevalensi {y.mean()*100:5.2f}% | "
          f"Bantul {int((X.wilayah=='Bantul').sum()):,} / "
          f"Sleman {int((X.wilayah=='Sleman').sum()):,}")

In [ ]:
# ===================== BAGI DATA LATIH & UJI =====================
# Stratifikasi HANYA pada y, sama seperti notebook wilayah. Proporsi wilayah
# di dalam bagian latih/uji ikut diperiksa di bawah supaya ketimpangan wilayah
# yang tidak disengaja tidak lolos diam-diam.
ringkas = []
for nama, d in data.items():
    X_tr, X_te, y_tr, y_te = train_test_split(
        d["X"], d["y"], test_size=0.2, stratify=d["y"],
        random_state=RANDOM_STATE)
    d.update({"X_train": X_tr, "X_test": X_te, "y_train": y_tr, "y_test": y_te})

    ringkas.append({
        "Dataset": nama,
        "Rentang usia": d["usia"],
        "N total": f"{len(d['X']):,}",
        "N latih": f"{len(X_tr):,}",
        "N uji": f"{len(X_te):,}",
        "Prevalensi (%)": round(d["base_rate"] * 100, 2),
        "Kelas positif": "minoritas" if d["base_rate"] < 0.5 else "MAYORITAS",
        "% Sleman latih": round((X_tr.wilayah == "Sleman").mean() * 100, 2),
        "% Sleman uji": round((X_te.wilayah == "Sleman").mean() * 100, 2),
    })

display(pd.DataFrame(ringkas))

## 2. Definisi Fitur -- di Sinilah Perbedaannya

Notebook wilayah memakai enam fitur. Notebook ini memakai **tujuh**: `wilayah`
ditambahkan ke daftar fitur kategorik.

Kolom `id` tetap dibuang seperti biasa. Ia diacak oleh BPJS untuk melindungi data
peserta asli, jadi tidak membawa informasi apa pun -- dan setelah penggabungan ia
menjadi lebih tidak bermakna lagi, karena kedua wilayah memakai rentang nilai yang
sama sehingga satu nilai `id` kini menunjuk ke dua kelompok catatan yang tidak
berhubungan.

Kolom kategori BMI (`Kategori_BMI_WHO8`, `Kategori_BMI_WHO4`) juga tetap dibuang.
Keduanya diturunkan dari IMT yang sudah ada sebagai fitur numerik; menyertakannya
tidak menaikkan performa dan justru merusak pembacaan kepentingan fitur.

In [ ]:
# ===================== DEFINISI FITUR =====================
NUMERIC_COLS = ["Usia", "Tinggibadan", "Berat Badan", "IMT_recalculated"]
CATEG_COLS = ["Jeniskelamin", "segmen", "wilayah"]     # <- 'wilayah' ditambahkan
FEATURES = CATEG_COLS + NUMERIC_COLS

print(f"Fitur kategorik : {CATEG_COLS}")
print(f"Fitur numerik   : {NUMERIC_COLS}")
print(f"Total           : {len(FEATURES)} kolom "
      f"(notebook wilayah memakai {len(FEATURES) - 1})")

In [ ]:
# ===================== PIPELINE PREPROCESSING =====================
def build_preprocessor():
    return ColumnTransformer([
        ("num", StandardScaler(), NUMERIC_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="first",
                              sparse_output=False), CATEG_COLS),
    ])


contoh_pre = build_preprocessor().fit(data["TBC"]["X_train"][FEATURES])
NAMA_FITUR = list(contoh_pre.get_feature_names_out())

print(f"{len(FEATURES)} kolom asli  ->  {len(NAMA_FITUR)} kolom setelah encoding\n")
for f in NAMA_FITUR:
    print("  ", f)

`drop="first"` berarti `wilayah` menjadi **satu** kolom biner, `wilayah_Sleman`.
Bantul adalah kategori acuan. Jadi koefisien atau kepentingan kolom itu terbaca
sebagai "seberapa jauh Sleman berbeda dari Bantul, setelah yang lain
diperhitungkan".

## 3. Pembangun Model dan Ruang Pencarian

Persis sama dengan notebook wilayah, termasuk ruang hyperparameter tiap model.
Satu-satunya yang berubah adalah `FEATURES` di atas.

In [ ]:
# ===================== PEMBANGUN MODEL & RUANG PENCARIAN =====================
def build_estimator(model_key, pakai_bobot=True, pos_weight=1.0):
    pre = build_preprocessor()
    bobot = "balanced" if pakai_bobot else None
    spw = pos_weight if pakai_bobot else 1.0

    if model_key == "dummy":
        # Pembanding dasar: menebak sesuai proporsi kelas, mengabaikan semua
        # fitur. Sengaja TIDAK menerima penyeimbang apa pun.
        return Pipeline([("pre", pre),
                         ("clf", DummyClassifier(strategy="prior"))]), {}

    if model_key == "logistic":
        clf = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE,
                                 class_weight=bobot)
        grid = {"clf__C": [0.001, 0.01, 0.1, 1, 10, 100],
                "clf__solver": ["lbfgs", "liblinear"]}

    elif model_key == "decision_tree":
        clf = DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight=bobot)
        grid = {"clf__criterion": ["gini", "entropy"],
                "clf__max_depth": [3, 5, 7, 10, 15, None],
                "clf__min_samples_split": [2, 10, 50],
                "clf__min_samples_leaf": [1, 5, 20]}

    elif model_key == "xgboost":
        clf = XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                            tree_method="hist", random_state=RANDOM_STATE,
                            n_jobs=1, scale_pos_weight=spw)
        grid = {"clf__n_estimators": [200, 300, 500],
                "clf__max_depth": [3, 4, 6, 8],
                "clf__learning_rate": [0.03, 0.05, 0.1],
                "clf__subsample": [0.8, 1.0],
                "clf__colsample_bytree": [0.8, 1.0],
                "clf__min_child_weight": [1, 3, 5]}
    else:
        raise ValueError(f"model_key tidak dikenal: {model_key}")

    return Pipeline([("pre", pre), ("clf", clf)]), grid


def penyeimbang_bawaan(base_rate):
    # Seimbangkan HANYA bila kelas positif memang minoritas. Pada Hipertensi
    # kelas positif adalah mayoritas (60%+), dan pembobotan di sana justru
    # menurunkan Accuracy maupun F1.
    return base_rate < 0.5


def latih_model(nama, model_key):
    d = data[nama]
    X_tr, X_te, y_tr = d["X_train"][FEATURES], d["X_test"][FEATURES], d["y_train"]

    pakai_bobot = penyeimbang_bawaan(d["base_rate"])
    pos_w = float((y_tr == 0).sum() / max((y_tr == 1).sum(), 1))
    pipe, grid = build_estimator(model_key, pakai_bobot, pos_w)

    if grid:
        search = RandomizedSearchCV(pipe, grid, n_iter=N_ITER, scoring="roc_auc",
                                    cv=CV, random_state=RANDOM_STATE, n_jobs=-1)
        search.fit(X_tr, y_tr)
        model, best_params = search.best_estimator_, search.best_params_
    else:
        model, best_params = pipe.fit(X_tr, y_tr), {}

    return model, model.predict_proba(X_te)[:, 1], best_params


for nama, d in data.items():
    print(f"{nama:<18} prevalensi {d['base_rate']*100:5.2f}%  ->  "
          f"class_weight: {penyeimbang_bawaan(d['base_rate'])}")

## 4. Melatih Seluruh Model

Enam belas model: empat penyakit x empat algoritma. Dengan `QUICK_MODE = False`
proses ini memakan waktu cukup lama -- data gabungan hampir dua kali lipat data
satu wilayah.

In [ ]:
# ===================== LATIH SEMUA MODEL =====================
MODEL_KEYS = ["dummy", "logistic", "decision_tree", "xgboost"]

models, probs, params = {}, {}, {}
t0 = time.time()

for nama, d in data.items():
    print(f"\n{nama}   |   prevalensi {d['base_rate']*100:.1f}%   "
          f"|   class_weight: {penyeimbang_bawaan(d['base_rate'])}")
    for mk in MODEL_KEYS:
        t = time.time()
        mdl, p, bp = latih_model(nama, mk)
        models[(nama, mk)], probs[(nama, mk)], params[(nama, mk)] = mdl, p, bp
        print(f"   {mk:<15} selesai dalam {time.time()-t:>6.1f} detik")

print(f"\nTotal waktu pelatihan: {(time.time()-t0)/60:.1f} menit")

In [ ]:

# ===================== HYPERPARAMETER TERPILIH =====================
baris_hp = []
for (nama, mk), bp in params.items():
    if bp:
        baris_hp.append({"Dataset": nama, "Model": mk,
                         **{k.replace("clf__", ""): v for k, v in bp.items()}})

display(pd.DataFrame(baris_hp).set_index(["Dataset", "Model"]).sort_index())

## 5. Hasil dan Pemilihan Model Terbaik

In [ ]:
# ===================== EVALUASI =====================
def evaluasi(nama, model_key):
    y_te = data[nama]["y_test"]
    prob = probs[(nama, model_key)]
    pred = (prob >= 0.5).astype(int)
    return {
        "Model": model_key,
        "Accuracy": accuracy_score(y_te, pred),
        "Precision": precision_score(y_te, pred, zero_division=0),
        "Recall": recall_score(y_te, pred, zero_division=0),
        "F1-Score": f1_score(y_te, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_te, prob),
    }


def model_terbaik(tab):
    # SATU definisi "model terbaik", dipakai di seluruh notebook supaya tabel,
    # confusion matrix dan ekspor tidak mungkin menunjuk model yang berbeda.
    #
    # `dummy` DIKECUALIKAN. Pada Hipertensi kelas positif adalah mayoritas,
    # sehingga dummy mencetak F1 lebih tinggi daripada model sungguhan mana pun;
    # tanpa pengecualian ini idxmax akan memilihnya.
    kandidat = tab[tab["Model"] != "dummy"]
    return kandidat.loc[kandidat["F1-Score"].idxmax(), "Model"]


hasil = {}
for nama in data:
    tab = pd.DataFrame([evaluasi(nama, mk) for mk in MODEL_KEYS])
    hasil[nama] = tab
    terbaik = model_terbaik(tab)
    print(f"\n=== {nama}  (pita usia {data[nama]['usia']}, "
          f"prevalensi {data[nama]['base_rate']*100:.2f}%) ===")
    display(tab.round(4))
    print(f"   model terbaik (dummy dikecualikan): {terbaik}")

In [ ]:
# ===================== RINGKASAN LINTAS PENYAKIT =====================
baris_terbaik = []
for nama, tab in hasil.items():
    mk = model_terbaik(tab)
    r = tab[tab["Model"] == mk].iloc[0]
    baris_terbaik.append({
        "Penyakit": nama,
        "Pita usia": data[nama]["usia"],
        "Prevalensi (%)": round(data[nama]["base_rate"] * 100, 2),
        "Model terbaik": mk,
        **{k: round(r[k], 4) for k in
           ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]},
    })

tabel_terbaik = pd.DataFrame(baris_terbaik)
display(tabel_terbaik)

assert "dummy" not in set(tabel_terbaik["Model terbaik"]), \
    "dummy terpilih sebagai model terbaik -- ini bug, bukan hasil."
print("Pemeriksaan lulus: dummy tidak terpilih pada satu penyakit pun.")

In [ ]:
# ===================== CONFUSION MATRIX MODEL TERBAIK =====================
fig, axes = plt.subplots(2, 2, figsize=vz.FIG_2X2)

for ax, (nama, tab) in zip(axes.ravel(), hasil.items()):
    mk = model_terbaik(tab)            # fungsi yang SAMA dengan tabel di atas
    y_te = data[nama]["y_test"]
    pred = (probs[(nama, mk)] >= 0.5).astype(int)
    cm = confusion_matrix(y_te, pred)
    ConfusionMatrixDisplay(cm, display_labels=["Tidak Berisiko", "Berisiko"]).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format=",")
    ax.set_title(f"{nama} -- {mk}\n(pita usia {data[nama]['usia']})", fontsize=11)

fig.suptitle("Confusion Matrix Model Terbaik -- Data Gabungan", fontsize=13)
plt.tight_layout()
plt.show()

## 6. Apakah `wilayah` Berpengaruh?

Pertanyaan utama notebook ini. Kepentingan fitur dikelompokkan kembali ke fitur
aslinya lebih dulu -- `segmen` pecah menjadi enam kolom setelah encoding, dan
tanpa pengelompokan setiap pecahannya tampak kecil sehingga peringkatnya bisa
terbalik.

XGBoost dipakai untuk **keempat** penyakit di bagian ini, meskipun pada beberapa
penyakit model terbaiknya bukan XGBoost. Itu disengaja: kepentingan fitur hanya
bisa dibandingkan antar penyakit bila algoritmanya sama.

In [ ]:
# ===================== KEPENTINGAN FITUR, DIKELOMPOKKAN =====================
def kelompok_fitur(nama_kolom):
    # "num__Usia"            -> "Usia"
    # "cat__segmen_BP PN"    -> "segmen"
    # "cat__wilayah_Sleman"  -> "wilayah"
    if nama_kolom.startswith("num__"):
        return nama_kolom[len("num__"):]
    sisa = nama_kolom[len("cat__"):]
    for c in CATEG_COLS:
        if sisa.startswith(c + "_"):
            return c
    return sisa


kepentingan = {}
for nama in data:
    mdl = models[(nama, "xgboost")]
    nama_kolom = list(mdl.named_steps["pre"].get_feature_names_out())
    gain = mdl.named_steps["clf"].feature_importances_

    s = pd.Series(gain, index=nama_kolom).groupby(kelompok_fitur).sum()
    kepentingan[nama] = (s / s.sum()).reindex(FEATURES).fillna(0)

tabel_kep = pd.DataFrame(kepentingan).T[FEATURES]
display(tabel_kep.round(4))

print("Fitur paling berpengaruh per penyakit (gain, dikelompokkan):")
for nama in tabel_kep.index:
    s = tabel_kep.loc[nama].sort_values(ascending=False)
    print(f"   {nama:<18} {s.index[0]:<18} {s.iloc[0]:.4f}   "
          f"(disusul {s.index[1]} {s.iloc[1]:.4f})")

print("\nKepentingan fitur 'wilayah':")
for nama in tabel_kep.index:
    v = tabel_kep.loc[nama, "wilayah"]
    peringkat = int(tabel_kep.loc[nama].rank(ascending=False)["wilayah"])
    print(f"   {nama:<18} {v:.4f}   peringkat {peringkat} dari {len(FEATURES)}")

In [ ]:
# ===================== GRAFIK KEPENTINGAN FITUR =====================
fig, axes = plt.subplots(2, 2, figsize=vz.FIG_2X2)

for ax, nama in zip(axes.ravel(), tabel_kep.index):
    s = tabel_kep.loc[nama].sort_values()
    warna = ["#C96F0F" if f == "wilayah" else "#153763" for f in s.index]
    ax.barh(s.index, s.values, color=warna)
    ax.set_title(f"{nama}  (pita usia {data[nama]['usia']})", fontsize=11)
    ax.set_xlabel("Kepentingan (gain, dikelompokkan, dinormalkan)")
    for i, v in enumerate(s.values):
        ax.text(v + 0.008, i, f"{v:.3f}", va="center", fontsize=8)
    ax.set_xlim(0, max(s.values) * 1.18)

fig.suptitle("Kepentingan Fitur XGBoost -- 'wilayah' disorot oranye", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ===================== ARAH PENGARUH WILAYAH =====================
# Kepentingan gain tidak bertanda, jadi ia tidak memberi tahu wilayah MANA yang
# lebih berisiko. Koefisien regresi logistik memberi arah itu -- dan karena
# 'wilayah' hanya punya satu kolom (Bantul = acuan), tandanya langsung terbaca.
arah = []
for nama in data:
    mdl = models[(nama, "logistic")]
    kolom = list(mdl.named_steps["pre"].get_feature_names_out())
    koef = pd.Series(mdl.named_steps["clf"].coef_[0], index=kolom)
    k = koef["cat__wilayah_Sleman"]
    arah.append({
        "Penyakit": nama,
        "Pita usia": data[nama]["usia"],
        "Koefisien wilayah_Sleman": round(float(k), 4),
        "Odds ratio": round(float(np.exp(k)), 3),
        "Arah": "Sleman lebih tinggi" if k > 0 else "Sleman lebih rendah",
    })

tabel_arah = pd.DataFrame(arah)
display(tabel_arah)

print("Pembanding -- selisih prevalensi mentah (dari notebook 1):")
for nama, (_, target, _) in DISEASES.items():
    d = data[nama]["X"].assign(y=data[nama]["y"])
    p = d.groupby("wilayah")["y"].mean().mul(100).round(2)
    print(f"   {nama:<18} Bantul {p['Bantul']:5.2f}%  Sleman {p['Sleman']:5.2f}%  "
          f"selisih {p['Sleman'] - p['Bantul']:+.2f} poin")

Bacalah dua tabel terakhir bersama-sama.

**Kolom `Odds ratio`** menyatakan berapa kali lipat peluang seseorang ditandai
berisiko bila ia diskrining di Sleman alih-alih Bantul, **setelah** usia, jenis
kelamin, segmen kepesertaan dan antropometrinya disamakan. Nilai 1,00 berarti
kabupaten tidak menambah apa pun.

Bandingkan arahnya dengan selisih prevalensi mentah yang dicetak di bawahnya.
Bila arah keduanya sama, selisih antar wilayah **tidak** habis dijelaskan oleh
perbedaan komposisi -- masih ada sesuatu yang melekat pada wilayahnya sendiri.
Bila arahnya berbalik, artinya komposisi bukan hanya menjelaskan selisihnya
tetapi menutupi selisih yang berlawanan arah.

⚠️ "Sesuatu yang melekat pada wilayahnya" **bukan** berarti perbedaan biologis
antar penduduk kabupaten. Kandidat penjelasan yang jauh lebih mungkin: perbedaan
cara petugas mengisi kuesioner, perbedaan siapa yang diundang skrining, dan
faktor risiko yang memang tidak ada di data ini (status merokok yang paling
menonjol). Fitur `wilayah` menampung semua itu sekaligus dan tidak bisa
memisahkannya.

## 7. Ekspor Hasil

Formatnya identik dengan `ringkasan_model.csv` tiap wilayah, sehingga notebook
`4. Perbandingan Model dan Transfer Lintas Wilayah` dan `Dashboard/train_export.py`
dapat membacanya dengan kode yang sama.

In [ ]:
# ===================== EKSPOR ringkasan_model.csv =====================
baris = []
for nama, tab in hasil.items():
    for _, r in tab.iterrows():
        baris.append({
            "Dataset": nama,
            "Rentang usia": data[nama]["usia"],
            "Prevalensi": data[nama]["base_rate"],
            "Model": r["Model"],
            "Accuracy": r["Accuracy"],
            "Precision": r["Precision"],
            "Recall": r["Recall"],
            "F1-Score": r["F1-Score"],
            "ROC-AUC": r["ROC-AUC"],
        })

gabungan = pd.DataFrame(baris)
out = DATA_DIR / "ringkasan_model.csv"
gabungan.to_csv(out, index=False, encoding="utf-8-sig")

print(f"Tersimpan: {out}")
print(f"{len(gabungan)} baris ({len(data)} penyakit x {len(MODEL_KEYS)} model)")
display(gabungan.round(4))

# Skema harus sama persis dengan ringkasan_model.csv wilayah, kalau tidak
# notebook 4 dan dashboard akan gagal membacanya.
acuan = list(cfg.baca_ringkasan("Bantul").columns)
assert list(gabungan.columns) == acuan, \
    f"Skema berbeda!\n  gabungan: {list(gabungan.columns)}\n  Bantul  : {acuan}"
print("\nPemeriksaan lulus: skema identik dengan ringkasan_model.csv wilayah.")

In [ ]:

# ===================== EKSPOR HYPERPARAMETER TERPILIH =====================
# Ditulis ke berkas supaya Dashboard/train_export.py TIDAK perlu menyalin
# angka-angka ini dengan tangan. Nilai yang salah ketik tetap menghasilkan
# model yang jalan, hanya metriknya yang diam-diam berbeda dari yang dilaporkan.
import json as _json
from datetime import datetime as _dt

_hp = {
    "dibuat": _dt.now().isoformat(timespec="seconds"),
    "wilayah": "Gabungan",
    "random_state": RANDOM_STATE,
    "quick_mode": QUICK_MODE,
    "n_iter": N_ITER,
    "cv_folds": CV_FOLDS,
    "fitur": FEATURES,
    "penyakit": {},
}

for _nama, _tab in hasil.items():
    _mk = model_terbaik(_tab)
    _hp["penyakit"][_nama] = {
        "model_terbaik": _mk,
        "penyeimbang": ("class_weight"
                        if penyeimbang_bawaan(data[_nama]["base_rate"]) else None),
        "pita_usia": data[_nama]["usia"],
        "prevalensi": float(data[_nama]["base_rate"]),
        "params": {k.replace("clf__", ""): v
                   for k, v in params[(_nama, _mk)].items()},
    }

_out_hp = DATA_DIR / "hyperparameter_terpilih.json"
_out_hp.write_text(_json.dumps(_hp, indent=1, ensure_ascii=False), encoding="utf-8")

print(f"Tersimpan: {_out_hp}")
for _nama, _v in _hp["penyakit"].items():
    print(f"   {_nama:<18} {_v['model_terbaik']:<15} "
          f"penyeimbang={_v['penyeimbang']}  {_v['params']}")

## 8. Ringkasan

Yang dihasilkan notebook ini:

- **`Gabungan/Data Setiap Target/ringkasan_model.csv`** -- 16 baris, skema sama
  dengan kedua wilayah. Menjadi masukan notebook 4 dan `Dashboard/train_export.py`.
- **Jawaban atas pertanyaan `wilayah`** di Bagian 6.

### Yang harus ikut disebut ketika hasil ini dilaporkan

**Diabetes tetap bukan prediksi.** Angka performa Diabetes yang tinggi di sini
sama seperti di notebook wilayah: model menyusun ulang aturan `IMT > 25` yang
dijalankan kuesioner. Menggabungkan dua kabupaten tidak mengubah hal itu.

**Presisi Kanker Paru dan TBC tetap rendah.** Sebagian besar catatan yang
ditandai berisiko oleh kedua model itu sebenarnya tidak berisiko. Itu konsekuensi
yang disengaja dari memilih recall tinggi pada penyakit langka, bukan kegagalan
yang bisa ditutupi.

**Model gabungan bukan otomatis lebih baik daripada model wilayah.** Ia dilatih
pada data lebih banyak, tetapi juga pada populasi yang lebih heterogen. Notebook
4 membandingkan ketiganya secara langsung -- termasuk menguji apakah model yang
dilatih di satu kabupaten masih berguna di kabupaten lain.